# Investigating the effect of different kernel size on SAL

SAL is particularly useful in smoothing nuisance variables (which is one kind of transformation). Given a different kernel sizes (receptive fields are another kind of transformation), the goal is to understand how SAL behaves and it's impact in understanding an image <br>

**Goal:** Quantify how convolutional kernel size (which changes the model’s effective receptive field) interacts with SAL-style local marginalization (anti-aliased downsampling) to shape the invariance–selectivity trade-off under small nuisance transformations (tiny shifts/rotations/scale) and occlusion. <br>

**What will be tested:** Three matched-capacity CNNs (3×3, 5×5, 7×7 kernels) × two pooling modes (standard vs. anti-aliased/SAL-ish). We’ll measure:
* Feature stability under small group actions $g$ (tiny translation/rotation/scale).
* Near-miss accuracy on confusable shape pairs.
* Occlusion sensitivity vs occlusion % and location.

## Dataset + Pre-processing

I am using the Kaggle dataset of geometric shapes [linked here](https://www.kaggle.com/datasets/reevald/geometric-shapes-mathematics).

In [18]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image, ImageDraw
import math, random
import numpy as np
import torch.backends.cudnn as cudnn
import torch.optim as optim
from tqdm import tqdm

In [11]:
SEED = 3
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
cudnn.deterministic = True
cudnn.benchmark = False

Data Loading

In [12]:
IMG_SIZE = 224 # staying at native resolution (to avoid additional transformations)

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # applies the nuisance transformations
    transforms.RandomAffine(
        degrees = 5,
        translate = (0.05, 0.05),
        scale = (0.95, 1.05),
        fill=0
    ),
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_dir = "dataset/train"
test_dir = "dataset/test"
val_dir = "dataset/val"

train_data = datasets.ImageFolder(
    root = train_dir,
    transform = train_transforms
)
test_data = datasets.ImageFolder(
    root = test_dir,
    transform = test_transforms
)
val_data = datasets.ImageFolder(
    root = val_dir,
    transform = test_transforms
)

# Creating DataLoaders
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2)

print("Classes: ", train_data.classes)
print("Number of training samples: ", len(train_data))

Classes:  ['circle', 'kite', 'parallelogram', 'rectangle', 'rhombus', 'square', 'trapezoid', 'triangle']
Number of training samples:  12000


## Model Initialization

In [13]:
import torch.nn as nn
import torch.nn.functional as F

BlurPoolDown is responsible for applying SAL. When the SAL option is set to True, the model builds a 5x5 Gaussian Kernel (will be changed for different Kernel Sizes in the future), applies the blur to each channel, and downsamples by x2 using average pooling.

This is as mentioned by Soatto/Chiuso - before you downsample, you blur (low-pass filter) the feature map, you marginalize out small spatial variations. The local averaging to remove nuisance variability before subsampling.

In [14]:
class BlurPoolDown(nn.Module):
    def __init__(self, channels):
        super().__init__()
        base = torch.tensor([1, 4, 6, 4, 1], dtype=torch.float32)
        kernel = (base[:, None] @ base[None, :])
        kernel = kernel / kernel.sum()
        kernel = kernel[None, None, :, :].repeat(channels, 1, 1, 1)
        self.register_buffer("kernel", kernel)
        self.groups = channels
    
    def forward(self, x):
        x = F.conv2d(x, self.kernel, stride=1, padding=2, groups=self.groups)
        return F.avg_pool2d(x, kernel_size=2, stride=2)

The ConvStage class is the one convolutional processing stage. It applies a Conv2D layer with kernel size K and normalizes it with BatchNorm2d. It then uses the ReLU activation function and applies SAL if the setting is set to True.

By changing the K value and toggling the SAL setting, you can learn how kernel size affects local invariance/selectivity and how SAL affects robustness to nuisance transformations

In [15]:
class ConvStage(nn.Module):
    def __init__(self, c_in, c_out, K=3, sal=False, downsample=True):
        super().__init__()
        self.conv = nn.Conv2d(c_in, c_out, K, padding=K//2, bias=False)
        self.bn   = nn.BatchNorm2d(c_out)
        self.act  = nn.ReLU(inplace=True)
        self.down = BlurPoolDown(c_out) if (sal and downsample) else (nn.AvgPool2d(2) if downsample else nn.Identity())
    def forward(self, x):
        x = self.act(self.bn(self.conv(x)))
        x = self.down(x)
        return x

KernelNet is the full network comprised of 4 ConvStages. Each ConvStage increases the number of feature channels, so for a 7x7 kernel the FLOPs per conv grows. Therefore, KernelNet implements a scale factor to reduce the channel width to optimize computational costs

In [16]:
class KernelNet(nn.Module):
    def __init__(self, num_classes, K=3, sal=False):
        super().__init__()
        # Widths scaled ~ 3/K to keep FLOPs roughly comparable across K
        scale = 3.0 / K
        w1, w2, w3, w4 = [max(16, int(v*scale)) for v in (64, 128, 192, 256)]

        self.s1 = ConvStage(3,   w1, K=K, sal=sal, downsample=True)   # 224 -> 112
        self.s2 = ConvStage(w1,  w2, K=K, sal=sal, downsample=True)   # 112 -> 56
        self.s3 = ConvStage(w2,  w3, K=K, sal=sal, downsample=True)   # 56 -> 28
        self.s4 = ConvStage(w3,  w4, K=K, sal=sal, downsample=True)   # 28 -> 14

        self.head = nn.Sequential(
            nn.Conv2d(w4, max(64, int(128*scale)), 1, bias=False),
            nn.BatchNorm2d(max(64, int(128*scale))),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(max(64, int(128*scale)), num_classes)

    def forward_features(self, x):
        x = self.s1(x); x = self.s2(x); x = self.s3(x); x = self.s4(x)
        x = self.head(x).flatten(1)
        return x

    def forward(self, x):
        f = self.forward_features(x)
        return self.fc(f)

Instantiating different instance combinations

In [17]:
num_classes = len(train_data.classes)
models = {
    "k3_max": KernelNet(num_classes, K=3, sal=False),
    "k5_max": KernelNet(num_classes, K=5, sal=False),
    "k7_max": KernelNet(num_classes, K=7, sal=False),
    "k3_sal": KernelNet(num_classes, K=3, sal=True),
    "k5_sal": KernelNet(num_classes, K=5, sal=True),
    "k7_sal": KernelNet(num_classes, K=7, sal=True),
}